# 1. Why Use Preprocessing Pipelines?

### Theory
A Machine Learning Pipeline chains sequential data transformations together with an estimator into a single, unified execution graph.

### Business Impact
In production deployment, raw user inputs must pass through identical transformation steps before hitting the inference model. Pipelines enforce seamless repeatability between model training and live production APIs.

### Risks
Applying manual data transformations separately across training and testing data leads to code duplication, missing transformation steps in production, and catastrophic data leakage.

### Decision Rules
- **Rule:** Wrap all transformers and estimators into a Scikit-Learn `Pipeline` or `ColumnTransformer` before executing cross-validation or fitting models.

In [1]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Load clean dataset
df = pd.read_csv("Cleaned_Validated_Data.csv")

print("=== Raw Pipeline Dataset Overview ===")
display(df.head(3))

=== Raw Pipeline Dataset Overview ===


,CustomerID,Age,Gender,TenureYears,MonthlyCharges,TotalCharges,ContractType,PaymentMethod,Churn
0,CUST-1000,34.0,Male,3,29.85,1098.216202,Two year,Credit card,No
1,CUST-1001,150.0,Female,10,56.95,4544.186090,month to month,Mailed check,Yes
2,CUST-1002,52.0,M,2,105.50,6144.766598,One year,Credit card,No


# 2. Pipeline vs. ColumnTransformer

### Theory
- **`Pipeline`:** Chains transformations sequentially on a single block of features (e.g., Imputation $\rightarrow$ Scaling).
- **`ColumnTransformer`:** Applies different transformation pipelines in parallel to specific feature subsets (e.g., continuous vs. categorical features).

### Business Impact
Allows complex multi-modal business data (pricing numbers, customer text tags, transaction timestamps) to be processed cleanly in a single execution step.

### Risks
Passing categorical columns into a numerical scaling pipeline causes crashes or erroneous metric calculations.

### Decision Rules
- Use `Pipeline` for sequential operations on a homogeneous data type.
- Use `ColumnTransformer` at the top level to direct feature groups to their respective sub-pipelines.

In [2]:
# Define numerical sub-pipeline
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Define categorical sub-pipeline
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

print("Sub-pipelines configured successfully.")

Sub-pipelines configured successfully.


# 3. Combining Transformations

### Theory
`ColumnTransformer` aggregates heterogeneous sub-pipelines, applies them to specified column lists, and concatenates the resulting feature arrays into a unified matrix.

### Business Impact
Automates feature engineering workflows for incoming raw production payloads.

### Risks
Column ordering shifts if feature names are hardcoded without explicitly referencing dynamic column lists.

### Decision Rules
- Explicitly separate numerical and categorical column lists using `select_dtypes()`.

In [3]:
# Separate features and target
X = df.drop(columns=["CustomerID", "Churn"])
y = np.where(df["Churn"] == "Yes", 1, 0)

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

# Combine into master ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

print("Numerical Columns:", num_cols)
print("Categorical Columns:", cat_cols)

Numerical Columns: ['Age', 'TenureYears', 'MonthlyCharges', 'TotalCharges']
Categorical Columns: ['Gender', 'ContractType', 'PaymentMethod']


C:\Users\abarn\AppData\Local\Temp\ipykernel_800\1542486877.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns.tolist()


# 4. fit(), transform(), and fit_transform() Mechanics

### Theory
- **`fit()`:** Calculates parameters (e.g., mean $\mu$, std $\sigma$, median, categories) from training data.
- **`transform()`:** Applies learned parameters to transform target data.
- **`fit_transform()`:** Computes parameters and transforms data in a single optimized pass.

### Business Impact
Saves processing time and guarantees operational consistency across environments.

### Risks
Calling `fit()` or `fit_transform()` on validation or test sets causes severe data leakage.

### Decision Rules
- Call `fit_transform()` **strictly on the Training set**.
- Call `transform()` **strictly on Validation, Test, and unseen Production datasets**.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Correct fit_transform on Train, transform on Test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"X_train Processed Matrix Shape: {X_train_processed.shape}")
print(f"X_test Processed Matrix Shape: {X_test_processed.shape}")
print("Notebook 14 execution completed successfully!")

X_train Processed Matrix Shape: (808, 18)
X_test Processed Matrix Shape: (202, 18)
Notebook 14 execution completed successfully!
